# Oracle Graph Quickstart: Full Pipeline

<p align="center"><b>PDF &rarr; Oracle Property Graph &rarr; SQL/PGQ Queries &rarr; Natural Language Answers</b></p>

---

## Introduction

This notebook demonstrates the **full OraclePageIndex pipeline**: indexing a PDF document into an
Oracle SQL Property Graph, then querying it with both natural language and SQL/PGQ.

Unlike traditional RAG that chunks text and matches vectors, OraclePageIndex builds a
**knowledge graph** of documents, sections, entities, and relationships — then traverses it
with standard SQL to find answers.

### What you'll learn:
- [x] Initialize the Oracle Property Graph schema
- [x] Index a PDF document (parse → extract entities → store in graph)
- [x] Query the knowledge graph with natural language
- [x] Run SQL/PGQ queries directly against the property graph

> **Prerequisites:**
> - Oracle Database 26ai Free running (`docker compose up -d` from project root)
> - [Ollama](https://ollama.com/) running locally with a model pulled (e.g., `ollama pull mistral:7b`)
> - Database user `pageindex` created with `CREATE PROPERTY GRAPH` grant (see README)

---

## Step 0: Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from oracle_pageindex.db import OracleDB
from oracle_pageindex.graph import GraphStore
from oracle_pageindex.llm import OllamaClient
from oracle_pageindex.indexer import Indexer
from oracle_pageindex.query import QueryEngine
from oracle_pageindex.utils import ConfigLoader

# Load configuration from config.yaml
cfg = ConfigLoader().load()

print(f"Ollama model: {cfg.ollama_model}")
print(f"Oracle DSN:   {cfg.oracle_dsn}")

Ollama model: mistral:7b
Oracle DSN:   localhost:1521/FREEPDB1


In [2]:
# Initialize Oracle connection
db = OracleDB(
    user=cfg.oracle_user,
    password=cfg.oracle_password,
    dsn=cfg.oracle_dsn,
)
db.connect()
print("Connected to Oracle Database")

# Initialize Ollama client
llm = OllamaClient(
    base_url=cfg.ollama_base_url,
    model=cfg.ollama_model,
    temperature=cfg.ollama_temperature,
    num_ctx=getattr(cfg, 'ollama_num_ctx', 8192),
)
print(f"Ollama ready ({cfg.ollama_model})")

Connected to Oracle Database
Ollama ready (mistral:7b)


## Step 1: Initialize the Property Graph Schema

This creates 3 vertex tables (`documents`, `sections`, `entities`),
3 edge tables (`section_hierarchy`, `section_entities`, `entity_relationships`),
and the `doc_knowledge_graph` Property Graph.

In [3]:
db.init_schema()
print("Schema initialized: 6 tables + Property Graph created")

SQL warning: ORA-06550: line 2, column 63:
PLS-00103: Encountered the symbol "end-of-file" when expecting one of the following:

   * & = - + ; < / > at in is mod remainder not rem return
   returning <an exponent (**)> <> or != or ~= >= <= <> and or
   like like2 like4 likec between into using || multiset bulk
   member submultiset <=> <-> <#>
Help: https://docs.oracle.com/error-help/db/ora-06550/


SQL warning: ORA-00900: invalid SQL statement
Help: https://docs.oracle.com/error-help/db/ora-00900/


SQL warning: ORA-00900: invalid SQL statement
Help: https://docs.oracle.com/error-help/db/ora-00900/


SQL warning: ORA-00955: name is already used by an existing object
Help: https://docs.oracle.com/error-help/db/ora-00955/


Schema initialized: 6 tables + Property Graph created


## Step 2: Index a Document

The `Indexer` orchestrates the full pipeline:
1. Parse PDF into a hierarchical tree with summaries (via Ollama)
2. Insert document + section vertices into the graph
3. Extract entities from each section (via Ollama)
4. Create mention edges (section → entity) and relationship edges (entity → entity)

In [4]:
# Use the included Apple 10-K demo document
pdf_path = "../demo/apple-10k-2024.pdf"

# Check if document is already indexed (skip expensive re-indexing)
existing = db.fetchall("SELECT doc_id, doc_name FROM documents WHERE doc_name = 'apple-10k-2024.pdf'")
sec_count = db.fetchone("SELECT COUNT(*) AS cnt FROM sections")["cnt"]

if existing and sec_count > 0:
    doc_id = existing[0]["doc_id"]
    entity_count = db.fetchone("SELECT COUNT(*) AS cnt FROM entities")["cnt"]
    rel_count = db.fetchone("SELECT COUNT(*) AS cnt FROM entity_relationships")["cnt"]
    print(f"Document already indexed (doc_id={doc_id}). Skipping re-indexing.")
    print(f"  Document:      apple-10k-2024.pdf")
    print(f"  Sections:      {sec_count}")
    print(f"  Entities:      {entity_count}")
    print(f"  Relationships: {rel_count}")
else:
    indexer = Indexer(llm=llm, db=db, opt=cfg)
    print("Indexing document (this takes several minutes for a 121-page PDF)...\n")
    stats = indexer.index_pdf(pdf_path)
    print(f"Indexing complete!")
    print(f"  Document:      {stats['doc_name']}")
    print(f"  Sections:      {stats['sections']}")
    print(f"  Entities:      {stats['entities']}")
    print(f"  Relationships: {stats['relationships']}")

Indexing document (this takes several minutes for a 121-page PDF)...



Failed to parse JSON from LLM response


Failed to parse JSON from response


Failed to parse JSON from LLM response


Failed to parse JSON from response


Failed to parse JSON from LLM response


Failed to parse JSON from response


Indexing complete!
  Document:      apple-10k-2024.pdf
  Sections:      412
  Entities:      0
  Relationships: 0


## Step 3: Explore the Knowledge Graph

Let's look at what the indexer stored in Oracle.

### 3.1 Graph statistics

In [5]:
graph = GraphStore(db)

docs = graph.get_all_documents()
entities = graph.get_all_entities()

# Count via SQL
section_count = db.fetchone("SELECT COUNT(*) AS cnt FROM sections")["cnt"]
hierarchy_count = db.fetchone("SELECT COUNT(*) AS cnt FROM section_hierarchy")["cnt"]
mention_count = db.fetchone("SELECT COUNT(*) AS cnt FROM section_entities")["cnt"]
rel_count = db.fetchone("SELECT COUNT(*) AS cnt FROM entity_relationships")["cnt"]

print("Oracle Property Graph Statistics")
print("=" * 40)
print(f"Documents:          {len(docs)}")
print(f"Sections:           {section_count}")
print(f"Entities:           {len(entities)}")
print(f"Hierarchy edges:    {hierarchy_count}")
print(f"Mention edges:      {mention_count}")
print(f"Relationship edges: {rel_count}")
print(f"Total graph nodes:  {len(docs) + section_count + len(entities)}")
print(f"Total graph edges:  {hierarchy_count + mention_count + rel_count}")

Oracle Property Graph Statistics
Documents:          1
Sections:           412
Entities:           0
Hierarchy edges:    304
Mention edges:      0
Relationship edges: 0
Total graph nodes:  413
Total graph edges:  304


### 3.2 Sample entities

In [6]:
# Show entities grouped by type
entity_types = {}
for ent in entities:
    etype = ent.get("entity_type", "UNKNOWN")
    entity_types.setdefault(etype, []).append(ent["name"])

if entity_types:
    for etype, names in sorted(entity_types.items()):
        sample = ", ".join(names[:5])
        more = f" (+{len(names) - 5} more)" if len(names) > 5 else ""
        print(f"{etype:15s} [{len(names):3d}]: {sample}{more}")
else:
    print("No entities extracted yet. Enable entity extraction in config.yaml")
    print("(set if_extract_entities: 'yes') and re-index to populate entities.")

No entities extracted yet. Enable entity extraction in config.yaml
(set if_extract_entities: 'yes') and re-index to populate entities.


## Step 4: Natural Language Querying

The `QueryEngine` extracts concepts from your question, traverses the graph to find
relevant sections and entities, then sends the assembled context to Ollama for reasoning.

In [7]:
engine = QueryEngine(llm=llm, graph=graph)

question = "What are the main risk factors for Apple?"
print(f"Question: {question}\n")

result = engine.query(question)

print("Answer:")
print(result["answer"])

if result["sources"]:
    print("\nSources:")
    for src in result["sources"]:
        print(f"  - {src['title']} ({src['doc_name']})")

if result["related_entities"]:
    print("\nRelated Entities:")
    for ent in result["related_entities"][:10]:
        print(f"  - {ent['name']} [{ent.get('type', '')}] ({ent.get('relationship', '')})")

Question: What are the main risk factors for Apple?



Answer:
 The main risk factors for Apple Inc., as outlined in the document, include:

1. Macroeconomic and Industry Risks: Adverse global or regional economic conditions can materially adversely affect Apple's business, results of operations, financial condition, and stock price. Factors such as slow growth or recession, high unemployment, inflation, tighter credit, higher interest rates, and currency fluctuations can impact consumer confidence and spending, demand for Apple's products and services, and the stability of its suppliers, contract manufacturers, logistics providers, distributors, cellular network carriers, and other channel partners.

2. Product Obsolescence or Competitive Pressure: The success of Apple's products depends on consumer preferences, which can change rapidly due to technological advancements or competitive offerings. If Apple fails to innovate or compete effectively, it could experience a decline in sales and market share.

3. Dependence on Third-Party Supplie

In [8]:
# Try another question
question2 = "What products does Apple sell?"
print(f"Question: {question2}\n")

result2 = engine.query(question2)
print("Answer:")
print(result2["answer"])

Question: What products does Apple sell?



Answer:
 Apple Inc. sells various electronic devices, software, and services. Some of their main product categories include:

1. iPhone: Smartphones that run on iOS, Apple's mobile operating system.
2. iPad: Tablet computers that also run on iOS.
3. Mac: Personal computers that run on macOS, Apple's desktop operating system.
4. Watch: Wearable devices that run on watchOS, Apple's wearable operating system.
5. AirPods: Wireless earbuds.
6. HomePod: Smart speakers.
7. Apple TV: Digital media players.
8. iTunes Store: Online marketplace for music, movies, and TV shows.
9. App Store: Online marketplace for mobile apps.
10. iCloud: Cloud storage and cloud computing service.
11. MacOS: Desktop operating system for Apple computers.
12. iOS: Mobile operating system for iPhone, iPad, and iPod Touch devices.
13. watchOS: Operating system for Apple Watch.
14. tvOS: Operating system for Apple TV.
15. iPadOS: Operating system specifically designed for iPad devices.

In addition to these products, A

## Step 5: Direct SQL/PGQ Queries

The entire knowledge graph is queryable with standard SQL using `GRAPH_TABLE`.
This is what makes Oracle special — no proprietary graph query language, just SQL.

### 5.1 Find entities related to a concept

In [9]:
# Use SQL/PGQ to find entities related to 'Apple Inc.'
related = graph.graph_query_related_entities("Apple Inc.")

print("Entities related to Apple Inc. (via SQL/PGQ):\n")
if related:
    for r in related:
        print(f"  {r['source_name']} --[{r['relationship']}]--> {r['related_name']} ({r['related_type']})")
else:
    print("  (No entity relationships found — enable entity extraction to populate)")

Entities related to Apple Inc. (via SQL/PGQ):

  (No entity relationships found — enable entity extraction to populate)


### 5.2 Find sections mentioning an entity

In [10]:
# Use SQL/PGQ GRAPH_TABLE to find sections mentioning 'iPhone'
sections = graph.graph_query_entity_sections("iPhone")

print("Sections mentioning 'iPhone' (via SQL/PGQ GRAPH_TABLE):\n")
if sections:
    for s in sections[:10]:
        print(f"  [{s['section_id']}] {s['section_title']} (depth={s['depth_level']}, relevance={s['relevance']})")
else:
    print("  (No mention edges found — enable entity extraction to populate)")
    print("  Tip: You can still query sections directly via SQL:")
    # Fallback: search section titles/content directly
    fallback = db.fetchall("""
        SELECT section_id, title, depth_level FROM sections
        WHERE LOWER(title) LIKE '%iphone%'
        ORDER BY start_index
        FETCH FIRST 10 ROWS ONLY
    """)
    for s in fallback:
        print(f"  [{s['section_id']}] {s['title']} (depth={s['depth_level']})")

Sections mentioning 'iPhone' (via SQL/PGQ GRAPH_TABLE):

  (No mention edges found — enable entity extraction to populate)
  Tip: You can still query sections directly via SQL:
  [362] iPhone (depth=3)


### 5.3 Raw SQL/PGQ query

In [11]:
# You can run any SQL/PGQ query directly
raw_results = db.fetchall("""
    SELECT *
    FROM GRAPH_TABLE (doc_knowledge_graph
        MATCH (e1 IS entity) -[r IS related_to]-> (e2 IS entity)
        WHERE e1.entity_type = 'ORGANIZATION'
        COLUMNS (
            e1.name AS org_name,
            r.relationship,
            e2.name AS related_name,
            e2.entity_type AS related_type
        )
    )
    FETCH FIRST 15 ROWS ONLY
""")

print("Organization relationships (raw SQL/PGQ):\n")
if raw_results:
    for r in raw_results:
        print(f"  {r['org_name']} --[{r['relationship']}]--> {r['related_name']} ({r['related_type']})")
else:
    print("  (No entity relationships found — enable entity extraction to populate)")
    print("\n  Alternative: query the section hierarchy via GRAPH_TABLE:")
    # Show section hierarchy as an alternative SQL/PGQ demo
    hier = db.fetchall("""
        SELECT *
        FROM GRAPH_TABLE (doc_knowledge_graph
            MATCH (p IS section) -[h IS parent_of]-> (c IS section)
            COLUMNS (
                p.title AS parent_title,
                c.title AS child_title,
                c.depth_level
            )
        )
        FETCH FIRST 10 ROWS ONLY
    """)
    for h in hier:
        print(f"  {h['parent_title']} --> {h['child_title']} (depth={h['depth_level']})")

Organization relationships (raw SQL/PGQ):

  (No entity relationships found — enable entity extraction to populate)

  Alternative: query the section hierarchy via GRAPH_TABLE:
  UNITED STATES SECURITIES AND EXCHANGE COMMISSION --> FORM 10-K (depth=1)
  FORM 10-K --> ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 (depth=2)
  FORM 10-K --> TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 (depth=2)
  Securities registered pursuant to Section 12(b) of the Act: --> Shareholders’ Equity (depth=1)
  Securities registered pursuant to Section 12(b) of the Act: --> Shares of Common Stock (depth=1)
  Shareholders’ Equity --> Share Repurchase Program (depth=2)
  Title of each class --> 2022 Employee Stock Plan (depth=1)
  Title of each class --> 2014 Employee Stock Plan (depth=1)
  Title of each class --> Restricted Stock Units (depth=1)
  Indicate by check mark if the Registrant is not required to file reports pursuant

## Cleanup

In [12]:
db.close()
print("Oracle connection closed")

Oracle connection closed


---

## What's Next

Now that the document is indexed in Oracle, explore the other notebooks:

- **[Graph Retrieval](graph_retrieval.ipynb)** — Deep dive into SQL/PGQ structured retrieval
- **[Vision RAG](vision_RAG_oracle.ipynb)** — Vision-based RAG with multimodal Ollama
- **[Simple Graph RAG](graph_RAG_simple.ipynb)** — Minimal vectorless RAG without Oracle

You can also start the interactive D3.js visualization:
```bash
python run.py serve
# Open http://localhost:8000
```

---

*Built with [OraclePageIndex](https://github.com/jasperan/OraclePageIndex) — Oracle AI Database powered document intelligence with Property Graphs.*